# Training Metrics Comparison

Load `0_metrics.json` from multiple training runs and compare selected metrics.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# ── Configure your runs here ──────────────────────────────────────────────────
# Each entry: (label, path to the training folder containing 0_metrics.json)

RUNS: list[tuple[str, str]] = [
    ("run-1", "/home/mthill/techdays26/exp_20260223_17-44"),
    # ("run-2", "/home/mthill/techdays26/train_run_2026-02-23-15-29"),
    # ("run-3", "./train_run_2026-02-24-10-30"),
]

# ── Load ──────────────────────────────────────────────────────────────────────
runs: dict[str, list[dict]] = {}
for label, folder in RUNS:
    p = Path(folder) / "0_metrics.json"
    if not p.exists():
        print(f"WARNING: {p} not found, skipping '{label}'")
        continue
    with p.open() as f:
        runs[label] = json.load(f)
    print(f"Loaded '{label}': {len(runs[label])} entries from {p}")

print(f"\n{len(runs)} run(s) loaded.")

In [ ]:
# ── Helper: extract a metric as numpy arrays ─────────────────────────────────


def get_metric(
    data: list[dict], key: str, x_key: str = "step"
) -> tuple[np.ndarray, np.ndarray]:
    """Return (x, y) arrays for the given metric key."""
    x = np.array([m[x_key] for m in data])
    y = np.array([m[key] for m in data], dtype=np.float64)
    return x, y


def plot_metric(
    metric_key: str,
    *,
    x_key: str = "step",
    title: str | None = None,
    ylabel: str | None = None,
    xlabel: str | None = None,
    yscale: str = "linear",
    figsize: tuple[float, float] = (10, 4),
    filter_inf: bool = True,
):
    """Plot a single metric across all loaded runs."""
    fig, ax = plt.subplots(figsize=figsize)
    for label, data in runs.items():
        x, y = get_metric(data, metric_key, x_key)
        if filter_inf:
            mask = np.isfinite(y)
            x, y = x[mask], y[mask]
        ax.plot(x, y, marker=".", markersize=3, linewidth=1, label=label)
    ax.set_xlabel(xlabel or x_key)
    ax.set_ylabel(ylabel or metric_key)
    ax.set_title(title or metric_key)
    ax.set_yscale(yscale)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_metrics_grid(
    metric_keys: list[str],
    *,
    x_key: str = "step",
    ncols: int = 2,
    figsize_per_plot: tuple[float, float] = (6, 3.5),
    yscales: dict[str, str] | None = None,
    filter_inf: bool = True,
):
    """Plot multiple metrics in a grid of subplots."""
    yscales = yscales or {}
    n = len(metric_keys)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(figsize_per_plot[0] * ncols, figsize_per_plot[1] * nrows),
        squeeze=False,
    )
    for idx, key in enumerate(metric_keys):
        ax = axes[idx // ncols][idx % ncols]
        for label, data in runs.items():
            x, y = get_metric(data, key, x_key)
            if filter_inf:
                mask = np.isfinite(y)
                x, y = x[mask], y[mask]
            ax.plot(x, y, marker=".", markersize=2, linewidth=1, label=label)
        ax.set_title(key)
        ax.set_xlabel(x_key)
        ax.set_yscale(yscales.get(key, "linear"))
        ax.legend(fontsize="small")
        ax.grid(True, alpha=0.3)
    # Hide unused subplots
    for idx in range(n, nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)
    plt.tight_layout()
    plt.show()

## Loss

In [ ]:
plot_metric("loss", title="Training Loss (MSE)", ylabel="loss")

## Relative Weight Update ||dW|| / ||W||

In [ ]:
plot_metric(
    "rel_weight_update",
    title="Relative Weight Update  ||\u0394W|| / ||W||",
    ylabel="||\u0394W|| / ||W||",
    yscale="log",
)

## Value Function Statistics

In [ ]:
plot_metrics_grid(
    ["V_old_mean", "V_old_std", "V_old_abs_mean", "V_old_min", "V_old_max"],
    ncols=2,
)

## Weight and Gradient Norms

In [ ]:
plot_metrics_grid(
    ["W_norm", "delta_W_norm", "grad_mean", "grad_std"],
    ncols=2,
)

## Training Dynamics

In [ ]:
plot_metrics_grid(
    ["done_frac", "n_wins", "moves_left_mean", "lr"],
    ncols=2,
)

## Wall Time per Logging Interval

In [ ]:
plot_metric("wall_time_s", title="Wall Time per Interval", ylabel="seconds")

## Custom: Pick Any Metric

Available keys in each metrics entry:

In [ ]:
# Print all available metric keys
sample_run = next(iter(runs.values()))
print("Available metrics:")
for k in sample_run[0].keys():
    print(f"  - {k}")

In [ ]:
# Plot any metric by name:
# plot_metric("grad_nnz", title="Non-zero Gradient Entries")

---

# Arena Results Comparison

Load `step_X_arena_result.json` files from each run and plot win rates, draw rates, etc. over training steps.

In [ ]:
import re
from collections import defaultdict

# ── Load arena results for each run ──────────────────────────────────────────


def load_arena_results(folder: str) -> dict[int, list[dict]]:
    """Load all step_X_arena_result.json files from a folder.

    Returns: {step: list_of_aggregate_rows}
    """
    folder = Path(folder)
    results = {}
    for p in sorted(folder.glob("step_*_arena_result.json")):
        m = re.search(r"step_(\d+)_arena_result", p.name)
        if m is None:
            continue
        step = int(m.group(1))
        with p.open() as f:
            data = json.load(f)
        results[step] = data["result"]["aggregates"]
    return dict(sorted(results.items()))


# Load arena results for all configured runs
arena_runs: dict[str, dict[int, list[dict]]] = {}
for label, folder in RUNS:
    ar = load_arena_results(folder)
    if not ar:
        print(f"WARNING: No arena result files found in '{folder}', skipping '{label}'")
        continue
    arena_runs[label] = ar
    steps = sorted(ar.keys())
    print(f"Loaded '{label}': {len(ar)} arena snapshots at steps {steps}")

print(f"\n{len(arena_runs)} run(s) with arena results.")

In [ ]:
# ── Arena helpers ─────────────────────────────────────────────────────────────


def get_matchup_key(row: dict) -> str:
    """Human-readable matchup identifier (ignoring epsilon)."""
    return f"{row['agent_yellow']} vs {row['agent_red']}"


def list_matchups(
    arena_data: dict[int, list[dict]], epsilon_zero_only: bool = True
) -> list[str]:
    """List all unique matchups across all steps."""
    matchups = set()
    for rows in arena_data.values():
        for r in rows:
            if epsilon_zero_only and (
                r["epsilon_yellow"] != 0.0 or r["epsilon_red"] != 0.0
            ):
                continue
            matchups.add(get_matchup_key(r))
    return sorted(matchups)


def extract_arena_series(
    arena_data: dict[int, list[dict]],
    *,
    matchup_filter: str | None = None,
    agent_filter: str | None = None,
    epsilon_zero_only: bool = True,
    perspective: str = "ntuple",
) -> dict[str, dict[str, list]]:
    """Extract per-matchup time series from arena data.

    Args:
        arena_data: {step: [aggregate_rows]}
        matchup_filter: Only include matchups containing this substring
        agent_filter: Only include rows where this agent appears
        epsilon_zero_only: Only include rows where both epsilons are 0
        perspective: Agent ID to compute win/loss/draw rates for

    Returns: {matchup_key: {"steps": [...], "win_rate": [...], "loss_rate": [...],
              "draw_rate": [...], "games": [...]}}
    """
    series: dict[str, dict[str, list]] = defaultdict(
        lambda: {
            "steps": [],
            "win_rate": [],
            "loss_rate": [],
            "draw_rate": [],
            "games": [],
        }
    )

    for step, rows in sorted(arena_data.items()):
        for r in rows:
            if epsilon_zero_only and (
                r["epsilon_yellow"] != 0.0 or r["epsilon_red"] != 0.0
            ):
                continue

            key = get_matchup_key(r)

            if matchup_filter and matchup_filter not in key:
                continue
            if (
                agent_filter
                and agent_filter not in r["agent_yellow"]
                and agent_filter not in r["agent_red"]
            ):
                continue

            n = r["games"]
            if n == 0:
                continue

            # Compute rates from the perspective of the specified agent
            if r["agent_yellow"] == perspective:
                win_rate = r["yellow_wins"] / n
                loss_rate = r["red_wins"] / n
            elif r["agent_red"] == perspective:
                win_rate = r["red_wins"] / n
                loss_rate = r["yellow_wins"] / n
            else:
                # Neither side is the perspective agent; use yellow perspective
                win_rate = r["yellow_wins"] / n
                loss_rate = r["red_wins"] / n

            draw_rate = r["draws"] / n

            series[key]["steps"].append(step)
            series[key]["win_rate"].append(win_rate)
            series[key]["loss_rate"].append(loss_rate)
            series[key]["draw_rate"].append(draw_rate)
            series[key]["games"].append(n)

    return dict(series)


def plot_arena_single_run(
    run_label: str,
    metric: str = "win_rate",
    *,
    matchup_filter: str | None = None,
    agent_filter: str | None = None,
    perspective: str = "ntuple",
    title: str | None = None,
    ylabel: str | None = None,
    figsize: tuple[float, float] = (12, 5),
):
    """Plot an arena metric for a single run: one line per matchup."""
    if run_label not in arena_runs:
        print(f"Run '{run_label}' not found in arena_runs")
        return
    series = extract_arena_series(
        arena_runs[run_label],
        matchup_filter=matchup_filter,
        agent_filter=agent_filter,
        perspective=perspective,
    )
    fig, ax = plt.subplots(figsize=figsize)
    for matchup_key, s in sorted(series.items()):
        ax.plot(
            s["steps"],
            s[metric],
            marker="o",
            markersize=4,
            linewidth=1.5,
            label=matchup_key,
        )
    ax.set_xlabel("step")
    ax.set_ylabel(ylabel or metric)
    filter_desc = matchup_filter or agent_filter or ""
    ax.set_title(
        title
        or f"[{run_label}] {metric} ({perspective} perspective)"
        + (f" — filter: {filter_desc}" if filter_desc else "")
    )
    ax.set_ylim(-0.05, 1.05)
    ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.4)
    ax.legend(fontsize="small", bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_arena_compare_runs(
    metric: str = "win_rate",
    *,
    matchup_filter: str | None = None,
    agent_filter: str | None = None,
    perspective: str = "ntuple",
    aggregate: str = "mean",
    title: str | None = None,
    ylabel: str | None = None,
    figsize: tuple[float, float] = (12, 5),
):
    """Compare an arena metric across runs: one line per run (averaged over matching matchups)."""
    fig, ax = plt.subplots(figsize=figsize)

    for label, arena_data in arena_runs.items():
        series = extract_arena_series(
            arena_data,
            matchup_filter=matchup_filter,
            agent_filter=agent_filter,
            perspective=perspective,
        )
        if not series:
            continue
        # Aggregate across matchups at each step
        all_steps = sorted(set(s for vals in series.values() for s in vals["steps"]))
        agg_vals = []
        for step in all_steps:
            vals_at_step = []
            for vals in series.values():
                if step in vals["steps"]:
                    idx = vals["steps"].index(step)
                    vals_at_step.append(vals[metric][idx])
            if vals_at_step:
                agg_vals.append(
                    np.mean(vals_at_step)
                    if aggregate == "mean"
                    else np.median(vals_at_step)
                )
            else:
                agg_vals.append(np.nan)
        ax.plot(
            all_steps, agg_vals, marker="o", markersize=4, linewidth=1.5, label=label
        )

    ax.set_xlabel("step")
    ax.set_ylabel(ylabel or f"{aggregate}({metric})")
    filter_desc = matchup_filter or agent_filter or "all matchups"
    ax.set_title(
        title
        or f"Arena {metric} ({aggregate} over {filter_desc}, {perspective} perspective)"
    )
    ax.set_ylim(-0.05, 1.05)
    ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.4)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_arena_stacked(
    run_label: str,
    matchup_key: str,
    *,
    perspective: str = "ntuple",
    title: str | None = None,
    figsize: tuple[float, float] = (10, 4),
):
    """Stacked area chart: win / draw / loss for a single matchup in a single run."""
    if run_label not in arena_runs:
        print(f"Run '{run_label}' not found")
        return
    series = extract_arena_series(arena_runs[run_label], perspective=perspective)
    if matchup_key not in series:
        print(
            f"Matchup '{matchup_key}' not found. Available:\n"
            + "\n".join(f"  - {k}" for k in sorted(series.keys()))
        )
        return

    s = series[matchup_key]
    steps = np.array(s["steps"])
    wins = np.array(s["win_rate"])
    draws = np.array(s["draw_rate"])
    losses = np.array(s["loss_rate"])

    fig, ax = plt.subplots(figsize=figsize)
    ax.stackplot(
        steps,
        wins,
        draws,
        losses,
        labels=[f"{perspective} wins", "draws", f"{perspective} losses"],
        colors=["#2ecc71", "#95a5a6", "#e74c3c"],
        alpha=0.8,
    )
    ax.set_xlabel("step")
    ax.set_ylabel("fraction")
    ax.set_title(title or f"[{run_label}] {matchup_key} ({perspective} perspective)")
    ax.set_ylim(0, 1)
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# List all available matchups (from first run)
first_label = next(iter(arena_runs))
print(f"Matchups in '{first_label}':")
for m in list_matchups(arena_runs[first_label]):
    print(f"  - {m}")

## ntuple Win Rate — All Matchups (Single Run)

In [ ]:
# Win rate of ntuple across all matchups for the first run
plot_arena_single_run(first_label, "win_rate", perspective="ntuple")

## ntuple Win Rate — Filtered by Opponent

In [ ]:
# Filter to matchups involving "random" only
plot_arena_single_run(
    first_label, "win_rate", agent_filter="random", perspective="ntuple"
)

# Filter to matchups involving "1ply" opponents (no book)
plot_arena_single_run(
    first_label, "win_rate", matchup_filter="bitbully-1ply", perspective="ntuple"
)

## Compare Runs — Average Win Rate

In [ ]:
# Average win rate across all matchups, one line per run
plot_arena_compare_runs(
    "win_rate", perspective="ntuple", title="Mean ntuple Win Rate (all matchups)"
)

# Average win rate only against random agent
plot_arena_compare_runs(
    "win_rate",
    agent_filter="random",
    perspective="ntuple",
    title="Mean ntuple Win Rate (vs random)",
)

## Stacked Win/Draw/Loss for a Single Matchup

In [ ]:
# Stacked area: win/draw/loss breakdown for ntuple vs a specific opponent
# Change the matchup_key to any from the list above
plot_arena_stacked(first_label, "ntuple vs random", perspective="ntuple")
plot_arena_stacked(first_label, "ntuple vs bitbully-1ply", perspective="ntuple")

## Compare Runs — Draw Rate and Loss Rate

In [ ]:
plot_arena_compare_runs(
    "draw_rate", perspective="ntuple", title="Mean ntuple Draw Rate (all matchups)"
)
plot_arena_compare_runs(
    "loss_rate", perspective="ntuple", title="Mean ntuple Loss Rate (all matchups)"
)